## Import

In [1]:
import warnings
import sys
import platform
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
from tqdm import tqdm

from eegkit.models import TaskDTO, FilterParamsDTO, EpochParamsDTO
from notebook_utils import reload_classes, reload_data_classes, get_model

warnings.filterwarnings("ignore", message=".*boundary.*data discontinuities.*")
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive, and thus cannot be shown")

## Load

In [2]:
EEGSubjectData = reload_data_classes()
release = 1
data_dir = f'/mount/NAS-public-dataset/HBN-EEG/cmi_bids_R{release}'
subject_data = EEGSubjectData(data_dir)

In [3]:
EEGController, EEGUI = reload_classes()
controller = EEGController(subject_data)
subject_model = controller.subject_model
visualizer = controller.visualizer
data_service = controller.data_service

In [4]:
from eegkit.models import TaskDTO, FilterParamsDTO, EpochParamsDTO


def prepare_task_data(controller, subject_index=0, task_name="surroundSupp", run="1",
                      l_freq=3.0, h_freq=35.0, tmin=-0.2, tmax=2.4,
                      ch_list=None):
    """
    Prepare EEG task data for a given subject/task/run.

    Returns:
      task_model, task_dto, filter_params, epoch_params,
      raw, epochs, labels, X (numpy array of shape [n_epochs, n_channels, n_times])
    """

    # --- subject selection ---
    subjects = sorted(controller.list_subjects())
    if not subjects:
        raise ValueError("No subjects found!")
    if subject_index >= len(subjects):
        raise IndexError(f"subject_index {subject_index} out of range (max {len(subjects) - 1})")
    subject = subjects[subject_index]

    # --- task/run selection ---
    task_keys = sorted(controller.list_tasks(subject))
    matches = [(t, r) for t, r in task_keys if t == task_name]
    if not matches:
        raise ValueError(f"Task '{task_name}' not found for subject {subject}")

    selected = None
    if run is None:
        selected = matches[0]
    elif run.lower() == "all":
        for t, r in matches:
            if r and "all" in r.lower():
                selected = (t, r)
                break
    else:
        for t, r in matches:
            if r == run:
                selected = (t, r)
                break
    if selected is None:
        raise ValueError(f"Run '{run}' not found for task '{task_name}' in subject {subject}")

    task, run_val = selected

    # --- DTOs ---
    task_dto = TaskDTO(subject=subject, task=task, run=run_val)
    filter_params = FilterParamsDTO(l_freq=l_freq, h_freq=h_freq)
    epoch_params = EpochParamsDTO(l_freq=l_freq, h_freq=h_freq, tmin=tmin, tmax=tmax)

    # --- load model ---
    subject_model = controller.subject_model
    task_model = subject_model.get_task(task_dto)

    # --- get raw and epochs ---
    raw = task_model.get_filtered_raw(filter_params)
    epochs, labels = task_model.get_epochs(epoch_params)

    # --- pick channels (optional) ---
    if ch_list is not None:
        epochs = epochs.copy().pick(ch_list)

    X = epochs.get_data()  # numpy array (n_epochs, n_channels, n_times)

    return task_model, task_dto, filter_params, epoch_params, raw, epochs, labels, X


## NN Class

In [5]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(device)

cuda:1


In [6]:
class EEGNetMultiOutput(nn.Module):
    def __init__(self, n_classes=(2, 4, 3)):
        super().__init__()
        self.n_classes = n_classes

        self.conv1 = nn.Conv2d(1, 16, (1, 64))
        self.batchnorm1 = nn.BatchNorm2d(16, affine=False)

        self.padding1 = nn.ZeroPad2d((16, 17, 0, 1))
        self.conv2 = nn.Conv2d(16, 32, (2, 32))
        self.batchnorm2 = nn.BatchNorm2d(32, affine=False)
        self.pooling2 = nn.MaxPool2d((2, 4))

        self.padding2 = nn.ZeroPad2d((2, 1, 4, 3))
        self.conv3 = nn.Conv2d(32, 64, (8, 4))
        self.batchnorm3 = nn.BatchNorm2d(64, affine=False)
        self.pooling3 = nn.MaxPool2d((2, 4))

        # LazyLinear heads (infer in_features automatically)
        self.fc_bg = nn.LazyLinear(self.n_classes[0])
        self.fc_fg = nn.LazyLinear(self.n_classes[1])
        self.fc_stim = nn.LazyLinear(self.n_classes[2])

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)

        x = F.elu(self.conv1(x))
        x = self.batchnorm1(x)
        x = F.dropout(x, 0.25)

        x = self.padding1(x)
        x = F.elu(self.conv2(x))
        x = self.batchnorm2(x)
        x = F.dropout(x, 0.25)
        x = self.pooling2(x)

        x = self.padding2(x)
        x = F.elu(self.conv3(x))
        x = self.batchnorm3(x)
        x = F.dropout(x, 0.25)
        x = self.pooling3(x)

        x = torch.flatten(x, start_dim=1)

        out_bg = self.fc_bg(x)
        out_fg = self.fc_fg(x)
        out_stim = self.fc_stim(x)
        return out_bg, out_fg, out_stim

# SurroundSupp

In [7]:
ch_list = ["E70", "E71", "E74", "E75", "E76", "E81", "E82", "E83"]

task_model, task_dto, f_params, e_params, raw, epochs, labels, X = prepare_task_data(
    controller,
    subject_index=0,
    task_name="surroundSupp",
    run="All",
    l_freq=3.0, h_freq=35.0,
    tmin=-0.2, tmax=2.4,
    ch_list=ch_list
)

[CACHE HIT] Raw filtered found at /mount/NAS-workspace-portal/eeg2025-Vistec/.eegcache/sub-NDARAC904DMU/surroundSupp/run-All-2/rawfilt/ae733bdd9e1ba7ed-88e03614bab6bda2-v1_eeg.fif


/mount/NAS-workspace-portal/eeg2025-Vistec/eegkit/cache/cache_service.py:78: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(p.as_posix(), preload=True, verbose="ERROR")


[CACHE HIT] Epochs found at /mount/NAS-workspace-portal/eeg2025-Vistec/.eegcache/sub-NDARAC904DMU/surroundSupp/run-All-2/epochs/3331c7b8f6b09866-88e03614bab6bda2-v1_epo.fif


In [8]:
data_service.show_annotations(task_dto, f_params)

{'PowerLineFrequency': 60,
 'TaskName': 'surroundSupp',
 'EEGChannelCount': 129,
 'EEGReference': 'Cz',
 'RecordingType': 'continuous',
 'RecordingDuration': 486.256,
 'SamplingFrequency': 500,
 'SoftwareFilters': 'n/a'}

In [9]:
n_epochs, n_channels, n_times = X.shape
print("Epochs shape:", X.shape)
print("Labels example:", labels[:10])

Epochs shape: (128, 8, 1301)
Labels example: ['bg1_fg0.0_stim2', 'bg1_fg0.3_stim3', 'bg1_fg0.6_stim1', 'bg1_fg1.0_stim2', 'bg1_fg0.0_stim3', 'bg1_fg0.3_stim2', 'bg1_fg0.6_stim1', 'bg1_fg1.0_stim3', 'bg1_fg0.0_stim2', 'bg1_fg0.3_stim3']


# Dependent 

In [10]:
def parse_label(label_str):
    bg = int(label_str.split("_")[0][-1])  # bg0/bg1 -> 0 or 1
    fg_map = {"0.0": 0, "0.3": 1, "0.6": 2, "1.0": 3}
    fg = fg_map[label_str.split("_")[1][2:]]  # fgX.X -> 0–3
    stim = int(label_str.split("_")[2][-1]) - 1  # stim1..3 -> 0–2
    return bg, fg, stim


def prepare_labels(labels):
    y_bg, y_fg, y_stim = zip(*(parse_label(l) for l in labels))
    return (torch.tensor(y_bg, dtype=torch.long),
            torch.tensor(y_fg, dtype=torch.long),
            torch.tensor(y_stim, dtype=torch.long))

In [11]:
def train_model(model, loader, device, epochs=50, lr=1e-3):
    """
    Train EEGNetMultiOutput model.

    model  : instance of EEGNetMultiOutput
    loader : DataLoader (X, y_bg, y_fg, y_stim)
    device : torch.device ("cuda" or "cpu")
    epochs : number of training epochs
    lr     : learning rate
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        correct_bg, correct_fg, correct_stim = 0, 0, 0
        total = 0

        progress = tqdm(loader, desc=f"Epoch {epoch:02d}", leave=False)
        for xb, yb_bg, yb_fg, yb_stim in progress:
            xb, yb_bg, yb_fg, yb_stim = (
                xb.to(device),
                yb_bg.to(device),
                yb_fg.to(device),
                yb_stim.to(device),
            )

            optimizer.zero_grad()
            out_bg, out_fg, out_stim = model(xb)

            loss = (criterion(out_bg, yb_bg)
                    + criterion(out_fg, yb_fg)
                    + criterion(out_stim, yb_stim))
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total += xb.size(0)

            correct_bg += (out_bg.argmax(1) == yb_bg).sum().item()
            correct_fg += (out_fg.argmax(1) == yb_fg).sum().item()
            correct_stim += (out_stim.argmax(1) == yb_stim).sum().item()

            progress.set_postfix(loss=loss.item())

        print(f"Epoch {epoch:02d} | "
              f"Loss {total_loss / total:.4f} | "
              f"BG acc {correct_bg / total:.3f} | "
              f"FG acc {correct_fg / total:.3f} | "
              f"Stim acc {correct_stim / total:.3f}")


In [12]:
# --- Prepare data ---
X_tensor = torch.tensor(X, dtype=torch.float32)  # (N, C, T) directly from MNE
y_bg, y_fg, y_stim = prepare_labels(labels)

dataset = TensorDataset(X_tensor, y_bg, y_fg, y_stim)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# --- Model, optimizer, loss ---
model = EEGNetMultiOutput().to(device)  # <-- new flexible version

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

/root/miniconda3/envs/py312_env/lib/python3.12/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [13]:
summary(model, input_size=X.shape)

Layer (type:depth-idx)                   Output Shape              Param #
EEGNetMultiOutput                        [128, 2]                  --
├─Conv2d: 1-1                            [128, 16, 8, 1238]        1,040
├─BatchNorm2d: 1-2                       [128, 16, 8, 1238]        --
├─ZeroPad2d: 1-3                         [128, 16, 9, 1271]        --
├─Conv2d: 1-4                            [128, 32, 8, 1240]        32,800
├─BatchNorm2d: 1-5                       [128, 32, 8, 1240]        --
├─MaxPool2d: 1-6                         [128, 32, 4, 310]         --
├─ZeroPad2d: 1-7                         [128, 32, 11, 313]        --
├─Conv2d: 1-8                            [128, 64, 4, 310]         65,600
├─BatchNorm2d: 1-9                       [128, 64, 4, 310]         --
├─MaxPool2d: 1-10                        [128, 64, 2, 77]          --
├─Linear: 1-11                           [128, 2]                  19,714
├─Linear: 1-12                           [128, 4]                  39,

In [14]:
train_model(model, loader, device, epochs=10)

Epoch 01 | Loss 16.5728 | BG acc 0.688 | FG acc 0.234 | Stim acc 0.359


Epoch 02 | Loss 8.3269 | BG acc 0.398 | FG acc 0.242 | Stim acc 0.359


Epoch 03 | Loss 7.0540 | BG acc 0.750 | FG acc 0.391 | Stim acc 0.375


Epoch 04 | Loss 4.3572 | BG acc 0.758 | FG acc 0.492 | Stim acc 0.484


Epoch 05 | Loss 3.8413 | BG acc 0.703 | FG acc 0.531 | Stim acc 0.531


Epoch 06 | Loss 2.6938 | BG acc 0.852 | FG acc 0.594 | Stim acc 0.617


Epoch 07 | Loss 2.2791 | BG acc 0.859 | FG acc 0.648 | Stim acc 0.609


Epoch 08 | Loss 1.9133 | BG acc 0.836 | FG acc 0.688 | Stim acc 0.680


Epoch 09 | Loss 1.3415 | BG acc 0.922 | FG acc 0.805 | Stim acc 0.766


Epoch 10 | Loss 1.3077 | BG acc 0.883 | FG acc 0.820 | Stim acc 0.797


In [15]:
validate_task_model, task_dto, f_params, e_params, raw, epochs, labels, validate_X = prepare_task_data(
    controller,
    subject_index=1,
    task_name="surroundSupp",
    run="All",
    l_freq=3.0, h_freq=35.0,
    tmin=-0.2, tmax=2.4,
    ch_list=ch_list
)

[CACHE HIT] Raw filtered found at /mount/NAS-workspace-portal/eeg2025-Vistec/.eegcache/sub-NDARAG143ARJ/surroundSupp/run-All-2/rawfilt/ae733bdd9e1ba7ed-fec9b0d6fb9a9604-v1_eeg.fif


/mount/NAS-workspace-portal/eeg2025-Vistec/eegkit/cache/cache_service.py:78: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(p.as_posix(), preload=True, verbose="ERROR")


[CACHE HIT] Epochs found at /mount/NAS-workspace-portal/eeg2025-Vistec/.eegcache/sub-NDARAG143ARJ/surroundSupp/run-All-2/epochs/3331c7b8f6b09866-fec9b0d6fb9a9604-v1_epo.fif


In [16]:
X_tensor = torch.tensor(validate_X, dtype=torch.float32).to(device)  # (N, C, T)
y_bg, y_fg, y_stim = prepare_labels(labels)
y_bg, y_fg, y_stim = y_bg.to(device), y_fg.to(device), y_stim.to(device)

# Criterion
criterion = nn.CrossEntropyLoss()

model.eval()
with torch.no_grad():
    out_bg, out_fg, out_stim = model(X_tensor)

    loss = (criterion(out_bg, y_bg)
            + criterion(out_fg, y_fg)
            + criterion(out_stim, y_stim)).item()

    acc_bg = (out_bg.argmax(1) == y_bg).float().mean().item()
    acc_fg = (out_fg.argmax(1) == y_fg).float().mean().item()
    acc_stim = (out_stim.argmax(1) == y_stim).float().mean().item()

In [17]:
print(f"Validation loss={loss:.4f} | BG={acc_bg:.3f}, FG={acc_fg:.3f}, Stim={acc_stim:.3f}")

Validation loss=3.5148 | BG=0.748, FG=0.220, Stim=0.331


# Contrast Change Detection classcification